# ReproduceMeON — LLM-Based Generation of Typed Semantic Links for Ontology Networks

This notebook reproduces the full pipeline:
1. RDF triple extraction from OWL ontology files
2. Context-sentence generation
3. DistilBERT fine-tuning (MLM, domain-specific)
4. Embedding generation
5. KMeans clustering + 90th-percentile cosine-similarity filtering
6. GPT-4o relationship generation with prompt engineering

> **Tip:** Steps 1–2 run on CPU. Steps 3–5 benefit from a GPU (CUDA).  
> Step 6 requires an OpenAI API key (`OPENAI_API_KEY` environment variable or set in the config cell below).

In [ ]:
# If openai import fails after this cell, restart the kernel and re-run from here.
!pip install --upgrade --quiet openai
!pip install --quiet 'transformers>=4.38' datasets torch pandas scikit-learn tqdm numpy
!pip install --quiet rdflib requests owlready2 matplotlib seaborn openpyxl

# Verify openai version
import importlib, subprocess
result = subprocess.run(['pip', 'show', 'openai'], capture_output=True, text=True)
for line in result.stdout.splitlines():
    if line.startswith('Version'):
        print(line)
        major = int(line.split('.')[0].split()[-1])
        if major < 1:
            print('WARNING: openai<1.0 detected. Restart the kernel, then re-run this cell.')
        else:
            print('openai>=1.0 confirmed — ready to proceed.')


## Configuration

In [ ]:
import os, json, re, logging
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import rdflib
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from scipy.stats import scoreatpercentile
from scipy.spatial.distance import pdist
from tqdm import tqdm
from rdflib import Graph, RDF, RDFS, OWL
from transformers import (
    EarlyStoppingCallback, DistilBertForMaskedLM, DistilBertTokenizer,
    DistilBertModel, DataCollatorForLanguageModeling, TrainingArguments, Trainer
)
from datasets import Dataset, DatasetDict
try:
    from openai import OpenAI
except ImportError:
    raise ImportError(
        "openai>=1.0 required. Run the install cell above, then "
        "restart the kernel (Kernel → Restart) and re-run from the top."
    )

# ─── Paths ────────────────────────────────────────────────────────────────────
DATA_DIR   = 'data'     # pre-provided ontologies, extracted triples, pre-computed CSVs
OUTPUT_DIR = 'output'   # pipeline writes intermediate files here
FIGS_DIR   = 'figures'  # optional: save cluster t-SNE plots here

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FIGS_DIR,   exist_ok=True)

# ─── OpenAI API key (Step 6 only) ─────────────────────────────────────────────
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', '')  # or paste key directly

# ─── Domain → ontology folder mapping ────────────────────────────────────────
DOMAIN_CONFIG = [
    ('Machine Learning',      os.path.join(DATA_DIR, 'ontologies', 'ML')),
    ('Computational',         os.path.join(DATA_DIR, 'ontologies', 'Computational')),
    ('Microscopy',            os.path.join(DATA_DIR, 'ontologies', 'Microscopy')),
    ('Experimental workflow', os.path.join(DATA_DIR, 'ontologies', 'Experiments')),
]

np.random.seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## Step 1: RDF Triple Extraction

Parses all OWL files in each domain folder and extracts (subject, predicate, object) triples together with `rdfs:label` and `rdfs:comment` annotations.  
**Pre-extracted CSVs are already in `data/extracted_triples/` — skip this step if you do not want to re-extract.**

In [ ]:
class OntologyParser:
    @staticmethod
    def extract_class_name(iri):
        return iri.split('#')[-1] if '#' in iri else iri.split('/')[-1]

    @staticmethod
    def extract_annotation(graph, iri, prop):
        return next(graph.objects(subject=iri, predicate=prop), None)

    @staticmethod
    def extract_from_folder(folder_path, domain):
        """Parse every .owl file in *folder_path* and return a DataFrame of triples."""
        import rdflib.namespace
        RDFS = rdflib.namespace.RDFS

        owl_files = [
            os.path.join(root, f)
            for root, _, files in os.walk(folder_path)
            for f in files if f.endswith('.owl')
        ]
        if not owl_files:
            print(f'  No OWL files found in {folder_path}')
            return pd.DataFrame()

        triples = []
        for path in owl_files:
            g = rdflib.Graph()
            try:
                g.parse(path, format=None)
                print(f'  Parsed: {os.path.basename(path)}')
            except Exception as e:
                print(f'  Skipped {os.path.basename(path)}: {e}')
                continue
            for subj, pred, obj in g:
                triples.append({
                    'Subject Class':   OntologyParser.extract_class_name(str(subj)),
                    'Subject Label':   str(OntologyParser.extract_annotation(g, subj, RDFS.label))   or None,
                    'Subject Comment': str(OntologyParser.extract_annotation(g, subj, RDFS.comment)) or None,
                    'Predicate':       OntologyParser.extract_class_name(str(pred)),
                    'Object Class':    OntologyParser.extract_class_name(str(obj)),
                    'Object Label':    str(OntologyParser.extract_annotation(g, obj, RDFS.label))    or None,
                    'Object Comment':  str(OntologyParser.extract_annotation(g, obj, RDFS.comment))  or None,
                    'Domain':          domain,
                    'Ontology Source': os.path.basename(path),
                })
        return pd.DataFrame(triples)

In [ ]:
# Run extraction for all domains and save per-domain CSVs.
# Skip this block if you want to use the pre-extracted files in data/extracted_triples/.

DOMAIN_TO_CSV = {
    'Machine Learning':      os.path.join(OUTPUT_DIR, 'rdf_triples_ml.csv'),
    'Computational':         os.path.join(OUTPUT_DIR, 'rdf_triples_computational.csv'),
    'Microscopy':            os.path.join(OUTPUT_DIR, 'rdf_triples_microscopy.csv'),
    'Experimental workflow': os.path.join(OUTPUT_DIR, 'rdf_triples_experiment.csv'),
}

for domain_name, folder in DOMAIN_CONFIG:
    print(f'\n[{domain_name}]')
    df = OntologyParser.extract_from_folder(folder, domain_name)
    out_path = DOMAIN_TO_CSV[domain_name]
    df.to_csv(out_path, index=False)
    print(f'  Saved {len(df)} triples → {out_path}')

## Step 2: Context Sentence Generation

Combines subject/predicate/object annotations into a natural-language sentence per triple. This sentence corpus is used to fine-tune DistilBERT.

> The pre-computed file `data/filtered_combined_sentences.csv` is already provided. Re-run this step only if you regenerated the extracted triples.

In [ ]:
class SentenceGenerator:
    TEMPLATE = (
        "The subject '{subject}' (labeled as '{subject_label}') described as '{subject_comment}' "
        "is related to the object '{object}' (labeled as '{object_label}') described as '{object_comment}' "
        "through the predicate '{predicate}' in the domain '{domain}'."
    )

    @classmethod
    def from_csv_files(cls, file_paths, template=None):
        """Load multiple RDF-triple CSVs and add a Sentence column."""
        template = template or cls.TEMPLATE
        required = [
            'Subject Class', 'Subject Label', 'Subject Comment',
            'Predicate', 'Object Class', 'Object Label', 'Object Comment',
            'Domain', 'Ontology Source'
        ]
        frames = []
        for path in file_paths:
            if not os.path.exists(path):
                print(f'  Not found: {path}')
                continue
            df = pd.read_csv(path, dtype=str, low_memory=False)
            missing = [c for c in required if c not in df.columns]
            if missing:
                print(f'  Skipping {path}: missing columns {missing}')
                continue
            df['Sentence'] = df.apply(
                lambda r: template.format(
                    subject=r['Subject Class'], subject_label=r['Subject Label'],
                    subject_comment=r['Subject Comment'], predicate=r['Predicate'],
                    object=r['Object Class'], object_label=r['Object Label'],
                    object_comment=r['Object Comment'], domain=r['Domain']
                ), axis=1
            )
            frames.append(df)
            print(f'  Processed {os.path.basename(path)} ({len(df)} rows)')
        return pd.concat(frames, ignore_index=True)

In [ ]:
# Build sentence corpus from pre-extracted CSVs
extracted_dir = os.path.join(DATA_DIR, 'extracted_triples')
csv_files = [
    os.path.join(extracted_dir, 'rdf_triples_ml.csv'),
    os.path.join(extracted_dir, 'rdf_triples_computational.csv'),
    os.path.join(extracted_dir, 'rdf_triples_microscopy.csv'),
    os.path.join(extracted_dir, 'rdf_triples_experiment.csv'),
]

sentences_df = SentenceGenerator.from_csv_files(csv_files)

# Filter to class-level triples only (subClassOf, equivalentClass)
keep_preds = {'subClassOf', 'equivalentClass', 'type'}
filtered_df = sentences_df[sentences_df['Predicate'].isin(keep_preds)].copy()
filtered_df = filtered_df.dropna(subset=['Sentence'])

out_sentences = os.path.join(OUTPUT_DIR, 'filtered_combined_sentences.csv')
filtered_df.to_csv(out_sentences, index=False)
print(f'Saved {len(filtered_df)} sentences → {out_sentences}')

# Quick stats
lengths = filtered_df['Sentence'].apply(lambda x: len(str(x).split()))
print(f'Avg sentence length: {lengths.mean():.1f} words | Max: {lengths.max()}')
print(filtered_df['Domain'].value_counts())

## Step 3: DistilBERT Fine-Tuning (MLM)

Fine-tunes `distilbert-base-uncased` on the sentence corpus using masked-language modelling (MLM). Training takes ~30 min on a T4 GPU with the default settings (25 epochs, batch=16).

> **Note:** The fine-tuned model is not included in the repository due to size. Run this step to produce it, or skip and use the pre-trained DistilBERT baseline in the evaluation notebook.

In [ ]:
os.environ['WANDB_MODE'] = 'disabled'

def prepare_dataset(sentences_df, tokenizer, split_ratios=(0.8, 0.1, 0.1)):
    dataset = Dataset.from_pandas(sentences_df[['Sentence']].dropna())
    tokenized = dataset.map(
        lambda ex: tokenizer(ex['Sentence'], truncation=True, padding='max_length', max_length=256),
        batched=True,
    )
    train_test = tokenized.train_test_split(test_size=split_ratios[1] + split_ratios[2], seed=42)
    val_test   = train_test['test'].train_test_split(
        test_size=split_ratios[2] / (split_ratios[1] + split_ratios[2]), seed=42
    )
    return DatasetDict({'train': train_test['train'], 'validation': val_test['train'], 'test': val_test['test']})


def fine_tune_distilbert(input_csv, output_dir, eval_steps=500, num_epochs=25, batch_size=16):
    model_name = 'distilbert-base-uncased'
    tokenizer  = DistilBertTokenizer.from_pretrained(model_name)
    model      = DistilBertForMaskedLM.from_pretrained(model_name)

    df = pd.read_csv(input_csv)
    tokenized_datasets = prepare_dataset(df, tokenizer)

    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True, mlm_probability=0.15)
    training_args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy='steps', eval_steps=eval_steps,
        save_steps=eval_steps, logging_steps=eval_steps,
        learning_rate=5e-5, per_device_train_batch_size=batch_size,
        num_train_epochs=num_epochs, warmup_steps=500, save_total_limit=2,
        fp16=torch.cuda.is_available(), load_best_model_at_end=True, report_to='none',
    )
    trainer = Trainer(
        model=model, args=training_args,
        train_dataset=tokenized_datasets['train'],
        eval_dataset=tokenized_datasets['validation'],
        data_collator=data_collator,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
    )
    trainer.train()
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)

    # Plot losses
    logs   = trainer.state.log_history
    t_loss = [l['loss']      for l in logs if 'loss'      in l]
    v_loss = [l['eval_loss'] for l in logs if 'eval_loss' in l]
    steps  = list(range(0, len(t_loss) * eval_steps, eval_steps))
    plt.figure(figsize=(10, 4))
    plt.plot(steps, t_loss, label='Train')
    plt.plot(steps[:len(v_loss)], v_loss, label='Validation')
    plt.xlabel('Steps'); plt.ylabel('Loss'); plt.title('Fine-tuning Loss'); plt.legend(); plt.tight_layout(); plt.show()

    test_metrics = trainer.evaluate(tokenized_datasets['test'])
    print('Test metrics:', test_metrics)
    return trainer

In [ ]:
FINE_TUNED_MODEL_DIR = os.path.join(OUTPUT_DIR, 'fine_tuned_distilbert')

trainer = fine_tune_distilbert(
    input_csv  = os.path.join(DATA_DIR, 'filtered_combined_sentences.csv'),
    output_dir = FINE_TUNED_MODEL_DIR,
    eval_steps = 500,
    num_epochs = 25,
    batch_size = 16,
)
print(f'Fine-tuned model saved to {FINE_TUNED_MODEL_DIR}')

## Step 4: Embedding Generation

Generates 768-dim embeddings (mean-pooled DistilBERT hidden state) for every concept's context sentence.

In [ ]:
class EmbeddingGenerator:
    def __init__(self, model_path):
        self.tokenizer = DistilBertTokenizer.from_pretrained(model_path)
        self.model     = DistilBertModel.from_pretrained(model_path).eval()
        self.device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model.to(self.device)

    def _embed_batch(self, texts, max_length=256):
        inputs = self.tokenizer(
            list(texts), return_tensors='pt', padding=True,
            truncation=True, max_length=max_length
        ).to(self.device)
        with torch.no_grad():
            return self.model(**inputs).last_hidden_state.mean(dim=1).cpu().numpy()

    def generate(self, df, batch_size=32):
        """Add Subject/Object Embedding columns to *df* in-place."""
        df = df.copy()
        df['Subject Context'] = df.apply(
            lambda r: f"{r['Subject Label']} - {r['Subject Comment']}"
            if pd.notna(r.get('Subject Label')) and pd.notna(r.get('Subject Comment'))
            else (str(r.get('Subject Label','')) or str(r.get('Subject Comment','')) or str(r['Subject Class'])),
            axis=1
        )
        df['Object Context'] = df.apply(
            lambda r: f"{r['Object Label']} - {r['Object Comment']}"
            if pd.notna(r.get('Object Label')) and pd.notna(r.get('Object Comment'))
            else (str(r.get('Object Label','')) or str(r.get('Object Comment','')) or str(r['Object Class'])),
            axis=1
        )
        for col, ctx_col in [('Subject Embedding', 'Subject Context'), ('Object Embedding', 'Object Context')]:
            print(f'Generating {col}...')
            embs = []
            for i in tqdm(range(0, len(df), batch_size)):
                embs.append(self._embed_batch(df[ctx_col].iloc[i:i+batch_size].fillna('')))
            df[col] = list(np.vstack(embs))
        return df

In [ ]:
sentences_csv = os.path.join(DATA_DIR, 'filtered_combined_sentences.csv')
sentences_df  = pd.read_csv(sentences_csv, dtype=str)

# Use fine-tuned model if available, else fall back to pre-trained
if os.path.isdir(FINE_TUNED_MODEL_DIR):
    model_path = FINE_TUNED_MODEL_DIR
    print('Using fine-tuned DistilBERT')
else:
    model_path = 'distilbert-base-uncased'
    print('Fine-tuned model not found — using pre-trained DistilBERT')

generator  = EmbeddingGenerator(model_path)
embedded_df = generator.generate(sentences_df, batch_size=32)

print(f'Embeddings generated for {len(embedded_df)} triples')
print(embedded_df['Domain'].value_counts())

## Step 5: KMeans Clustering + Cosine Similarity Filtering

Clusters embeddings, then retains only concept pairs whose cosine similarity exceeds the cluster-wise 90th-percentile threshold. **Pre-computed result: `data/filtered_similarity_scores_clusters.csv`.**

In [ ]:
class IntrinsicAnalysis:
    def __init__(self, n_clusters=4):
        self.n_clusters = n_clusters

    def find_optimal_clusters(self, embeddings, max_clusters=10):
        silhouette_scores = []
        for k in range(2, max_clusters + 1):
            km = KMeans(n_clusters=k, random_state=42)
            km.fit(embeddings)
            silhouette_scores.append(silhouette_score(embeddings, km.labels_))
        optimal_k = np.argmax(silhouette_scores) + 2
        plt.figure(figsize=(8, 4))
        plt.plot(range(2, max_clusters+1), silhouette_scores, marker='o')
        plt.xlabel('K'); plt.ylabel('Silhouette'); plt.title('Optimal Cluster Selection')
        plt.tight_layout(); plt.show()
        print(f'Optimal k = {optimal_k}')
        return optimal_k

    def cluster(self, df):
        embs = np.vstack(df['Subject Embedding'].tolist() + df['Object Embedding'].tolist())
        labels = KMeans(n_clusters=self.n_clusters, random_state=42).fit_predict(embs)
        df = df.copy()
        df['Subject Cluster'] = labels[:len(df)]
        df['Object Cluster']  = labels[len(df):]
        return df

    def visualize(self, df, save_dir=None):
        embs = np.vstack(df['Subject Embedding'].tolist() + df['Object Embedding'].tolist())
        tsne = TSNE(n_components=2, random_state=42)
        coords = tsne.fit_transform(embs)
        clusters = df['Subject Cluster'].tolist() + df['Object Cluster'].tolist()
        domains  = df['Domain'].tolist() * 2
        tsne_df  = pd.DataFrame({'x': coords[:,0], 'y': coords[:,1],
                                 'Cluster': clusters, 'Domain': domains})
        for c in range(self.n_clusters):
            sub = tsne_df[tsne_df['Cluster'] == c]
            plt.figure(figsize=(9,6))
            sns.scatterplot(data=sub, x='x', y='y', hue='Domain', palette='tab10', alpha=0.8)
            plt.title(f'Cluster {c} — t-SNE')
            plt.legend(bbox_to_anchor=(1.05,1))
            if save_dir:
                plt.savefig(os.path.join(save_dir, f'cluster_{c}.png'), dpi=150, bbox_inches='tight')
            plt.show()


class SimilarityMeasurement:
    @staticmethod
    def compute_dynamic_threshold(matrix):
        flat = matrix[np.triu_indices(matrix.shape[0], k=1)]
        return float(scoreatpercentile(flat, 90))

    @staticmethod
    def compute(df):
        rows = []
        for cluster, grp in df.groupby('Subject Cluster'):
            grp = grp.reset_index(drop=True)
            s_embs = np.vstack(grp['Subject Embedding'].tolist())
            o_embs = np.vstack(grp['Object Embedding'].tolist())
            sim    = cosine_similarity(s_embs, o_embs)
            thr    = SimilarityMeasurement.compute_dynamic_threshold(sim)
            print(f'Cluster {cluster}: {len(grp)} items, threshold={thr:.4f}')
            for i in tqdm(range(len(grp)), desc=f'Cluster {cluster}', leave=False):
                for j in range(len(grp)):
                    if i != j and sim[i,j] >= thr:
                        rows.append({
                            'Subject 1':       grp.iloc[i]['Subject Class'],
                            'Subject 2':       grp.iloc[j]['Object Class'],
                            'Domain 1':        grp.iloc[i]['Domain'],
                            'Domain 2':        grp.iloc[j]['Domain'],
                            'Ontology 1':      grp.iloc[i]['Ontology Source'],
                            'Ontology 2':      grp.iloc[j]['Ontology Source'],
                            'Similarity Score': sim[i,j],
                            'Cluster':         cluster,
                        })
        pairs = pd.DataFrame(rows)
        # Remove symmetric duplicates
        pairs['_key'] = pairs.apply(lambda r: tuple(sorted([r['Subject 1'],r['Subject 2']])), axis=1)
        pairs = pairs.drop_duplicates('_key').drop(columns='_key').dropna()
        return pairs

In [ ]:
all_embs = np.vstack(embedded_df['Subject Embedding'].tolist() + embedded_df['Object Embedding'].tolist())

analyzer = IntrinsicAnalysis()
optimal_k = analyzer.find_optimal_clusters(all_embs, max_clusters=10)
analyzer.n_clusters = optimal_k

clustered_df = analyzer.cluster(embedded_df)
print('Clustering done.')

# Optional: visualize clusters (saves PNGs to FIGS_DIR)
# analyzer.visualize(clustered_df, save_dir=FIGS_DIR)

print('Computing pairwise cosine similarities within clusters...')
similarity_df = SimilarityMeasurement.compute(clustered_df)
print(f'Retained {len(similarity_df):,} unique pairs after filtering')

sim_out = os.path.join(OUTPUT_DIR, 'filtered_similarity_scores_clusters.csv')
similarity_df.to_csv(sim_out, index=False)
print(f'Saved → {sim_out}')

print('\nSimilarity score summary:')
print(similarity_df['Similarity Score'].describe())

## Step 6: GPT-4o Relationship Generation

Queries GPT-4o for each sampled concept pair and asks it to:
- name a semantic relationship (`enables`, `requires`, etc.)
- classify its nature (Functional, Causal, Hierarchical, …)
- provide a justification sentence
- cite a supporting reference

**API key required.** Set `OPENAI_API_KEY` in the config cell, or pass via environment variable.

Due to API cost, the code samples up to `SAMPLES_PER_GROUP` pairs per domain-combination. Set `SAMPLES_PER_GROUP = None` to process all retained pairs (may incur significant cost).

In [ ]:
# ─── Sampling configuration ───────────────────────────────────────────────
SAMPLES_PER_GROUP = 30   # pairs sampled per domain-pair combination; set None for all
GPT_MODEL         = 'gpt-4o'
# ─────────────────────────────────────────────────────────────────────────

def generate_relationship(client, subject1, domain1, ontology1, subject2, domain2, ontology2, similarity):
    """Call GPT-4o and return a dict with Relationship, Nature, Source, Target, Justification, Reference."""
    system_msg = (
        'You are an expert in ontology construction and knowledge graph generation. '
        'Your task is to identify meaningful semantic relationships between concepts in the '
        'ReproduceMeON ontology network. '
        'Rules: avoid vague relations like "related to"; reflect real-world dependencies; '
        'provide a one-sentence justification; cite a supporting publication.'
    )
    user_msg = f"""Concept A: {subject1} (domain: {domain1}, ontology: {ontology1})
Concept B: {subject2} (domain: {domain2}, ontology: {ontology2})
Cosine similarity: {similarity:.4f}

Decide whether a meaningful semantic link exists between Concept A and Concept B.
If yes, return JSON:
{{
  \"Relationship\": \"<verb phrase>\",
  \"Relationship Nature\": \"<Hierarchical|Functional|Causal|Temporal|Spatial|Compositional|Comparative|Transformational|Instrumental>\",
  \"Source\": \"<Concept A or B>\",
  \"Target\": \"<Concept A or B>\",
  \"Justification Sentence\": \"<one sentence>\",
  \"Scientific Reference\": \"<citation>\"
}}
If no link exists, return JSON with \"Relationship\": \"No Relationship\" and explain in Justification Sentence.
Return ONLY valid JSON, no markdown fences."""

    try:
        resp = client.chat.completions.create(
            model=GPT_MODEL,
            messages=[{'role':'system','content':system_msg},{'role':'user','content':user_msg}],
            max_tokens=400, temperature=0.2
        )
        content = resp.choices[0].message.content.strip()
        # Strip markdown code fences if present
        content = re.sub(r'^```(?:json)?\n?', '', content)
        content = re.sub(r'\n?```$', '', content)
        result = json.loads(content)
        return {
            'Relationship':          result.get('Relationship', ''),
            'Relationship Nature':   result.get('Relationship Nature', ''),
            'Source':                result.get('Source', ''),
            'Target':                result.get('Target', ''),
            'Justification Sentence':result.get('Justification Sentence', ''),
            'Scientific Reference':  result.get('Scientific Reference', ''),
        }
    except Exception as e:
        return {'Relationship': f'Error: {e}', 'Relationship Nature':'',
                'Source':'','Target':'','Justification Sentence':'','Scientific Reference':''}

In [ ]:
# Load similarity pairs (use pre-computed file from data/ if pipeline output not available)
sim_path = os.path.join(OUTPUT_DIR, 'filtered_similarity_scores_clusters.csv')
if not os.path.exists(sim_path):
    sim_path = os.path.join(DATA_DIR, 'filtered_similarity_scores_clusters.csv')
    print(f'Using pre-computed similarity file: {sim_path}')
sim_df = pd.read_csv(sim_path)
print(f'Total candidate pairs: {len(sim_df):,}')

# Sample uniformly across domain combinations
grouped = sim_df.groupby(['Domain 1', 'Domain 2'])
if SAMPLES_PER_GROUP is not None:
    sample_df = grouped.apply(
        lambda g: g.sample(n=min(SAMPLES_PER_GROUP, len(g)), random_state=42)
    ).reset_index(drop=True)
else:
    sample_df = sim_df.copy()
print(f'Sampled {len(sample_df):,} pairs for GPT-4o generation')
print(sample_df.groupby(['Domain 1','Domain 2']).size().to_string())

# Determine link type (inter- or intra-domain)
sample_df['Link Type'] = sample_df.apply(
    lambda r: 'Intra' if r['Domain 1'] == r['Domain 2'] else 'Inter', axis=1
)

if not OPENAI_API_KEY:
    print('WARNING: OPENAI_API_KEY is not set. Skipping GPT-4o generation.')
else:
    client = OpenAI(api_key=OPENAI_API_KEY)

    results = []
    for _, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc='GPT-4o'):
        rel = generate_relationship(
            client,
            subject1=str(row['Subject 1']), domain1=str(row['Domain 1']), ontology1=str(row['Ontology 1']),
            subject2=str(row['Subject 2']), domain2=str(row['Domain 2']), ontology2=str(row['Ontology 2']),
            similarity=float(row['Similarity Score'])
        )
        results.append({**row.to_dict(), **rel})

    output_df = pd.DataFrame(results)
    gpt_out = os.path.join(OUTPUT_DIR, 'relationships_gpt4o.csv')
    output_df.to_csv(gpt_out, index=False)
    print(f'Saved {len(output_df)} generated relationships → {gpt_out}')

    valid = output_df[output_df['Relationship'] != 'No Relationship']
    print(f'Relationships found: {len(valid)} / {len(output_df)} ({100*len(valid)/len(output_df):.1f}%)')
    print('\nTop relationship types:')
    print(valid['Relationship'].value_counts().head(10))